# Lab 6 — MLflow Experiment Log

**Day 04 · Distance-Based ML & MLOps · Cisco AI/ML Training**

---

## Learning objectives

1. Train a KNN pipeline and compute **test accuracy**.
2. Log **parameters**, **metrics**, and the **model** to MLflow.
3. Persist runs in a local **SQLite** tracking store.
4. Write `metrics.json` and **track it with DVC** (`dvc init`, `dvc add`, `dvc status`).

> **Checkpoints:** accuracy ≈ **0.58** · k = **7** · `metrics.json` written · `output/metrics.json.dvc` created



## MLflow tracking in one slide

| Artifact | What gets logged |
|----------|------------------|
| **Parameters** | `model`, `k` — reproducible config |
| **Metrics** | `accuracy` — compare runs in the UI |
| **Model** | Serialized sklearn `Pipeline` — deploy or audit later |
| **metrics.json** | Flat file for **DVC** (`dvc add`) in the classroom demo |

```text
train → MLflow log → metrics.json → dvc add → metrics.json.dvc pointer
```

Lab 4 exposed the model via FastAPI; Lab 6 **versions** the training run so teams can compare experiments over time.

---

## 1. Load data and train KNN (k = 7)

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-04":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "lending-club" / "lending_club_sample.csv").is_file():
            GH_ROOT = parent
            break

OUTPUT_DIR = GH_ROOT / "hands-on" / "day-04" / "output"
DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]

df = pd.read_csv(GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv")
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

X = df[NUMERIC_FEATURES]
y = df["default"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

k = 7
model = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k)),
    ]
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"k: {k}")
print(f"test accuracy: {accuracy:.4f}")

This lab uses **k=7** (accuracy ≈ 0.58 on this split) — a deliberate choice to show logging a specific run, not necessarily the Lab 3 optimum (k=3).

---

## 2. Configure MLflow tracking

In [ ]:
import mlflow

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
mlflow_db = OUTPUT_DIR / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{mlflow_db.as_posix()}")
mlflow.set_experiment("cisco-aiml-day04-lending-club")

print(f"tracking URI: sqlite:///{mlflow_db.name}")
print(f"experiment: cisco-aiml-day04-lending-club")

---

## 3. Log the run

In [ ]:
with mlflow.start_run(run_name="knn-baseline") as run:
    mlflow.log_param("model", "KNeighborsClassifier")
    mlflow.log_param("k", k)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.sklearn.log_model(model, artifact_path="model")
    run_id = run.info.run_id

print("Lab 6 — MLflow experiment log")
print(f"run_id: {run_id}")
print(f"accuracy logged: {accuracy:.4f}")

---

## 4. Write metrics.json (DVC demo artifact)

In [ ]:
import json

from IPython.display import display

metrics_path = OUTPUT_DIR / "metrics.json"
metrics_payload = {
    "accuracy": round(accuracy, 4),
    "k": k,
    "run_id": run_id,
}
metrics_path.write_text(json.dumps(metrics_payload), encoding="utf-8")

print(f"metrics artifact (DVC demo): {metrics_path.name}")
print(f"full path: {metrics_path}")
display(pd.DataFrame([metrics_payload]))

---

## 5. Track metrics.json with DVC

**DVC** (Data Version Control) stores large files outside Git and keeps a small `.dvc` pointer in the repo.

| Step | Command | Purpose |
|------|---------|--------|
| Init | `dvc init --no-scm` | Enable DVC in `day-04/` (classroom; no Git required) |
| Add | `dvc add output/metrics.json` | Snapshot metrics + create `metrics.json.dvc` |
| Status | `dvc status` | Confirm workspace is clean |

MLflow logs **experiments**; DVC versions **artifact files** — complementary tools (Day 1 Lab 5).


In [ ]:
DAY_DIR = GH_ROOT / "hands-on" / "day-04"
METRICS_REL = "output/metrics.json"

import subprocess

if not (DAY_DIR / ".dvc").is_dir():
    proc = subprocess.run(
        ["dvc", "init", "--no-scm"],
        cwd=DAY_DIR,
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr or proc.stdout)
    print("dvc init: OK")

proc = subprocess.run(
    ["dvc", "add", METRICS_REL],
    cwd=DAY_DIR,
    capture_output=True,
    text=True,
)
if proc.returncode != 0:
    raise RuntimeError(proc.stderr or proc.stdout)
print(proc.stdout or "dvc add: OK")

status = subprocess.run(["dvc", "status"], cwd=DAY_DIR, capture_output=True, text=True)
print(status.stdout or status.stderr)

dvc_pointer = DAY_DIR / f"{METRICS_REL}.dvc"
print(f"DVC pointer: {dvc_pointer.name}")
assert dvc_pointer.is_file(), "metrics.json.dvc should exist"


---

## 6. Verify the metrics file

In [ ]:
loaded = json.loads(metrics_path.read_text(encoding="utf-8"))
print(loaded)
assert metrics_path.is_file()
assert loaded["k"] == k
assert loaded["run_id"] == run_id

---

## 7. Launch MLflow UI (instructor demo)

From `hands-on/day-04/output`:

```bash
mlflow ui --backend-store-uri sqlite:///output/mlflow.db
```

Open **http://127.0.0.1:5000** — compare parameters, metrics, and the saved model artifact.

Full setup: [mlflow-ui-demo-setup.md](../../../docs/mlflow-ui-demo-setup.md)

---

## 8. Day 04 recap

In [ ]:
recap = pd.DataFrame({
    "lab": ["1 Distance", "2 KNN k=5", "3 Choose k", "4 FastAPI", "5 FeatureTools", "6 MLflow"],
    "checkpoint": [
        "cosine ≈ 0.90",
        "acc ≈ 0.55",
        "best k=3, acc ≈ 0.59",
        "GET/POST 200",
        "shape (1000, 6)",
        f"acc ≈ {accuracy:.2f}, metrics.json",
    ],
})
display(recap)

---

## 9. Checkpoint summary

In [ ]:
assert k == 7
assert abs(accuracy - 0.58) < 0.02
assert metrics_path.is_file()
assert loaded["accuracy"] == round(accuracy, 4)
print(f"tracking db: {mlflow_db.name}")
assert (DAY_DIR / "output/metrics.json.dvc").is_file()
print("✓ All checkpoint assertions passed")

## Extension — log a second run (k=3) and compare

<!-- cisco-enrich-2026-06 -->

In [ ]:
import mlflow
import mlflow.sklearn

for k_try in [3, 7]:
    pipe_k = Pipeline([
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k_try)),
    ])
    pipe_k.fit(X_train, y_train)
    acc_k = accuracy_score(y_test, pipe_k.predict(X_test))
    with mlflow.start_run(run_name=f"knn-k{k_try}"):
        mlflow.log_param("k", k_try)
        mlflow.log_metric("accuracy", acc_k)
        print(f"logged k={k_try} accuracy={acc_k:.3f}")
print("Open MLflow UI to compare runs (see docs/mlflow-ui-demo-setup.md)")


---

## Reflection questions

1. What would you log additionally for a fraud model (Day 6)?
2. Why keep both MLflow **and** a flat `metrics.json` for DVC?
3. How would you promote the logged model to the Lab 4 FastAPI service?

**Previous:** [Lab 5 — FeatureTools auto FE](lab05_featuretools_auto_fe.ipynb)  
**Next:** [Day 05 — Unsupervised Learning](../day-05/README.md)